In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS viajes_aereos_plata;

-- 1. Dimensión Aviones
CREATE OR REPLACE TABLE viajes_aereos_plata.dim_aviones AS
SELECT 
    aircraft_code AS codigo_avion,
    model AS modelo_avion,
    range AS rango_alcance_km
FROM viajes_aereos_bronce.aircrafts_data;

-- 2. Dimensión Aeropuertos
CREATE OR REPLACE TABLE viajes_aereos_plata.dim_aeropuertos AS
SELECT 
    airport_code AS codigo_aeropuerto,
    airport_name AS nombre_aeropuerto,
    city AS ciudad,
    timezone AS zona_horaria
FROM viajes_aereos_bronce.airports_data;

-- 3. Dimensión Asientos
CREATE OR REPLACE TABLE viajes_aereos_plata.dim_asientos AS
SELECT 
    aircraft_code AS codigo_avion,
    seat_no AS numero_asiento,
    CASE 
        WHEN fare_conditions = 'Business' THEN 'Ejecutiva'
        WHEN fare_conditions = 'Economy' THEN 'Económica'
        WHEN fare_conditions = 'Comfort' THEN 'Confort'
        ELSE fare_conditions 
    END AS clase_tarifa
FROM viajes_aereos_bronce.seats;

-- 4. Hechos Reservas (Corregido con TRY_CAST)
CREATE OR REPLACE TABLE viajes_aereos_plata.fact_reservas AS
SELECT 
    book_ref AS codigo_reserva,
    TRY_CAST(book_date AS TIMESTAMP) AS fecha_reserva,
    CAST(total_amount AS DECIMAL(10,2)) AS monto_total_reserva
FROM viajes_aereos_bronce.bookings;

-- 5. Dimensión Pasajeros
CREATE OR REPLACE TABLE viajes_aereos_plata.dim_pasajeros_tiquetes AS
SELECT 
    ticket_no AS numero_tiquete,
    book_ref AS codigo_reserva,
    passenger_id AS id_pasajero
FROM viajes_aereos_bronce.tickets;

-- 6. Hechos Vuelos (Corregido con TRY_CAST)
CREATE OR REPLACE TABLE viajes_aereos_plata.fact_vuelos AS
SELECT 
    flight_id AS id_vuelo,
    flight_no AS numero_vuelo,
    TRY_CAST(scheduled_departure AS TIMESTAMP) AS salida_programada,
    TRY_CAST(scheduled_arrival AS TIMESTAMP) AS llegada_programada,
    departure_airport AS aeropuerto_origen,
    arrival_airport AS aeropuerto_destino,
    CASE 
        WHEN status = 'On Time' THEN 'A Tiempo'
        WHEN status = 'Delayed' THEN 'Retrasado'
        WHEN status = 'Departed' THEN 'En Vuelo'
        WHEN status = 'Arrived' THEN 'Aterrizado'
        WHEN status = 'Scheduled' THEN 'Programado'
        WHEN status = 'Cancelled' THEN 'Cancelado'
        ELSE status 
    END AS estado_vuelo,
    aircraft_code AS codigo_avion,
    TRY_CAST(actual_departure AS TIMESTAMP) AS salida_real,
    TRY_CAST(actual_arrival AS TIMESTAMP) AS llegada_real
FROM viajes_aereos_bronce.flights;

-- 7. Hechos Detalle Tiquetes
CREATE OR REPLACE TABLE viajes_aereos_plata.fact_detalle_tiquetes AS
SELECT 
    ticket_no AS numero_tiquete,
    flight_id AS id_vuelo,
    CASE 
        WHEN fare_conditions = 'Business' THEN 'Ejecutiva'
        WHEN fare_conditions = 'Economy' THEN 'Económica'
        WHEN fare_conditions = 'Comfort' THEN 'Confort'
        ELSE fare_conditions 
    END AS clase_tarifa,
    CAST(amount AS DECIMAL(10,2)) AS monto_tarifa
FROM viajes_aereos_bronce.ticket_flights;

-- 8. Hechos Pases Abordaje
CREATE OR REPLACE TABLE viajes_aereos_plata.fact_pases_abordaje AS
SELECT 
    ticket_no AS numero_tiquete,
    flight_id AS id_vuelo,
    boarding_no AS numero_abordaje,
    seat_no AS numero_asiento
FROM viajes_aereos_bronce.boarding_passes;


num_affected_rows,num_inserted_rows
